# `ptof_obs_hallucination_detection`

## What this notebook does
Answers "did the agent tell the truth about what it was given" -- the core of the agent
input/output relationship. Combines two independent signal layers into one risk verdict per
call: **Layer 2** (`ai_similarity` between response and prompt, scored incrementally via a
watermark), and **Layer 3** (regex-based ungrounded-token detection -- numbers/codes in the
response that don't appear in the prompt).

A prior Layer 1 (an LLM-judge `verify_grounding` capability's verdict) was removed as dead code:
`verify_grounding` never existed in this data, so it never produced a verdict.

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `04_hallucination_detection` -- runs in parallel with
  `02_latency_detection`, `03_malformed_output`, `05_behavioral_correlation`, after
  `01_bronze_projections`, before `06_alert`.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`),
  `capability_registry` (human-curated by `ptof_obs_setup_seed`, joined as
  `r.active = true AND r.is_generative = true` -- non-generative capabilities are structurally
  exempt from grounding checks), `_obs_watermark` (the incremental-scoring bookmark, seeded by
  `ptof_obs_setup_seed`).
- **Downstream:** `ptof_obs_alert.ipynb` reads `hallucination_signal` (v1 detector
  `hallucination_high`, CRITICAL) -- only 2 scored rows exist system-wide as of this writing,
  both `low` risk: armed and correct, but data-starved. Its value grows with call volume.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `capability_registry`, `_obs_watermark`,
  `faithfulness_scores`/`v_ungrounded_tokens` (this notebook's own outputs, joined together in
  the final cell).
- **Writes:** `faithfulness_scores` (Layer 2, incrementally MERGEd), `v_ungrounded_tokens`
  (Layer 3, a VIEW not a table -- always current), `hallucination_signal` (the combined verdict
  `ptof_obs_alert` reads), and `_obs_watermark` (advanced only after Layer 2 scoring succeeds).

In [ ]:
# Capture a single timestamp for both the MERGE upper bound and the watermark advance.
# This closes the gap where a row could arrive between two separate current_timestamp() calls
# (cell-2's MERGE and cell-5's watermark advance), be past the new watermark, but never in the
# MERGE source — permanently skipped from hallucination scoring.
_scoring_ts = spark.sql("SELECT current_timestamp() AS ts").first().ts
spark.conf.set("obs.scoring_ts", str(_scoring_ts))
print(f"scoring_ts = {_scoring_ts}")

In [ ]:
%sql
-- faithfulness_scores — Layer 2: ai_similarity(response, prompt) as a numeric grounding proxy,
-- scored incrementally past the watermark rather than rescanning all of v_llm_bronze every run
-- (the one place in this codebase using the watermark+MERGE incremental pattern, seeded by
-- ptof_obs_setup_seed's _obs_watermark table).
-- Layer 2. Scores only rows past the watermark; without this, ai_similarity is never invoked
-- and every new row lands in hallucination_signal as 'unverified'.
-- Upper bound uses ${obs.scoring_ts} (captured in the cell above) rather than
-- current_timestamp() to close the watermark gap -- see that cell's comment.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.faithfulness_scores (
    id STRING, shift_date STRING, shift_type STRING, batch_nbr STRING,
    capability STRING, called_at TIMESTAMP,
    resp_vs_prompt_similarity DOUBLE,
    similarity_pctile_in_capability DOUBLE
);

MERGE INTO mq_gmdf_dev.oil_obs.faithfulness_scores t
USING (
  SELECT b.id, b.shift_date, b.shift_type, b.batch_nbr, b.capability, b.called_at,
         ai_similarity(cast(b.response_parsed AS STRING),
                       cast(b.user_prompt     AS STRING)) AS resp_vs_prompt_similarity
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true AND r.is_generative = true AND r.is_groundable = true
  WHERE b.called_at > (SELECT last_processed_ts FROM mq_gmdf_dev.oil_obs._obs_watermark
                        WHERE detector = 'faithfulness_scores')
    AND b.called_at <= '${obs.scoring_ts}'::timestamp
    AND b.success = true
    AND b.is_blank_output        = false
    AND b.is_credential_fastfail = false
) s
ON t.id = s.id
WHEN NOT MATCHED THEN INSERT (id, shift_date, shift_type, batch_nbr, capability, called_at,
                              resp_vs_prompt_similarity)
  VALUES (s.id, s.shift_date, s.shift_type, s.batch_nbr, s.capability, s.called_at,
          s.resp_vs_prompt_similarity);

In [ ]:
%sql
-- v_ungrounded_tokens -- Layer 3: regex-based detection of specific numbers/codes that
-- appear in the response but not the prompt (a cheap, judge-free proxy for "the agent
-- invented a number"). A VIEW, not a table, so it's always current -- no incremental/
-- watermark bookkeeping needed since regex extraction is cheap relative to Layer 2's
-- ai_similarity calls.
-- v_ungrounded_tokens
-- Regex additions: [A-Z]{1,4}\d{3,} catches E007896, FILL3320, TT3990_REF3275A, QBMS3990 —
-- all of which the previous \b-anchored numeric pattern missed entirely.
-- Normalization: strips % and thousands separators, trims trailing .0, so prompt -205200 and
-- response -205,200 no longer read as ungrounded (confirmed false positive on dsa_compare).
-- Scoped to active + groundable capabilities via capability_registry join (defense-in-depth:
-- cell-7's hallucination_signal also filters, but running regex on out-of-scope traffic wastes
-- compute and the VIEW should be self-documenting about its scope).
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_ungrounded_tokens AS
WITH raw AS (
  SELECT
      b.id, b.shift_date, b.shift_type, b.batch_nbr, b.capability, b.called_at,
      regexp_extract_all(cast(b.response_parsed AS STRING),
        '(?i)\\b([A-Z]{1,4}\\d{3,}(?:_[A-Za-z0-9]+)*|\\d{1,7}(?:[.,]\\d+)?%?|\\d{1,2}:\\d{2}|\\d{4}-\\d{2}-\\d{2})\\b', 1) AS resp_raw,
      regexp_extract_all(cast(b.user_prompt AS STRING),
        '(?i)\\b([A-Z]{1,4}\\d{3,}(?:_[A-Za-z0-9]+)*|\\d{1,7}(?:[.,]\\d+)?%?|\\d{1,2}:\\d{2}|\\d{4}-\\d{2}-\\d{2})\\b', 1) AS prompt_raw
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true AND r.is_groundable = true
  WHERE b.success = true
    AND b.is_blank_output        = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 7 DAYS
),
norm AS (
  -- strip formatting differences (%, thousands separators, trailing .0) before comparing,
  -- so the same number formatted two ways in prompt vs. response isn't a false-positive
  -- mismatch.
  SELECT id, shift_date, shift_type, batch_nbr, capability, called_at,
         array_distinct(transform(resp_raw,   x ->
           regexp_replace(regexp_replace(x, '[%,]', ''), '\\.0+$', ''))) AS resp_tokens,
         array_distinct(transform(prompt_raw, x ->
           regexp_replace(regexp_replace(x, '[%,]', ''), '\\.0+$', ''))) AS prompt_tokens
  FROM raw
)
-- ungrounded_tokens: tokens present in the response but absent from the prompt -- what
-- hallucination_signal's ungrounded_token_count branch reads.
SELECT id, shift_date, shift_type, batch_nbr, capability, called_at,
       array_except(resp_tokens, prompt_tokens)       AS ungrounded_tokens,
       size(array_except(resp_tokens, prompt_tokens)) AS ungrounded_token_count
FROM norm
WHERE size(array_except(resp_tokens, prompt_tokens)) > 0;

In [0]:
%sql
-- Recomputes each row's similarity percentile WITHIN its own capability -- the number
-- hallucination_signal actually thresholds on (a fixed similarity cutoff would treat a
-- naturally-more-varied capability the same as a naturally-consistent one).
-- Percentiles are relative, so they shift as rows are added. MERGE rather than
-- CREATE OR REPLACE ... AS SELECT ... FROM itself, which drops and recreates the table.
MERGE INTO mq_gmdf_dev.oil_obs.faithfulness_scores t
USING (
  SELECT id,
         percent_rank() OVER (PARTITION BY capability ORDER BY resp_vs_prompt_similarity)
           AS pctile
  FROM mq_gmdf_dev.oil_obs.faithfulness_scores
) s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.similarity_pctile_in_capability = s.pctile;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
193,193,0,0


In [ ]:
%sql
-- Advances the incremental-scoring bookmark so the next run only scores rows newer than this
-- one. Kept as its own cell deliberately:
-- Separate cell on purpose: if scoring above fails, the watermark must NOT advance, so the next
-- run re-scans the same range.
-- Uses ${obs.scoring_ts} (captured before the MERGE) rather than current_timestamp() -- both
-- the MERGE upper bound and this advance now use the same timestamp, closing the gap where a
-- row could arrive between two separate current_timestamp() evaluations.
UPDATE mq_gmdf_dev.oil_obs._obs_watermark
SET last_processed_ts = '${obs.scoring_ts}'::timestamp,
    updated_at = current_timestamp()
WHERE detector = 'faithfulness_scores';

In [ ]:
%sql
-- hallucination_signal -- combines the two active signal layers into one hallucination_risk
-- verdict per call. This is what ptof_obs_alert's hallucination_high detector (CRITICAL in v1)
-- reads.
-- Gated on is_groundable: only capabilities whose output can meaningfully be grounded against
-- input data are admitted to the hallucination pipeline. This decouples "can we check it" from
-- "is it GxP-relevant" -- admission is about groundability, severity about compliance.
-- model_config included as attribution: per SME, "if a hallucination is detected THEN you dig
-- deeper into the specifics of the config."
-- Window: GREATEST(watermark, 7d) — prevents scanning rows older than the watermark that were
-- never in the MERGE source. Those rows have similarity IS NULL by construction (not because
-- scoring failed), which inflated hallucination_unverified_rate.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.hallucination_signal AS
SELECT
    b.id                        AS row_id,
    b.shift_date, b.shift_type, b.batch_nbr,
    b.capability                AS verified_capability,
    b.model_config,
    b.called_at,
    f.resp_vs_prompt_similarity,
    f.similarity_pctile_in_capability,
    d.ungrounded_token_count,
    -- hallucination_risk: the combined verdict, resolved via Layers 2/3 -- the
    -- percentile+similarity-floor branch, or the ungrounded-token-count branch for
    -- groundable+GxP-relevant capabilities.
    -- Rule 3 (ungrounded tokens) now requires a similarity cross-check: both regex AND semantic
    -- similarity must agree something is off before escalating. f.resp_vs_prompt_similarity IS
    -- NULL is fail-open (scoring wasn't available, so don't suppress).
    CASE
      WHEN d.ungrounded_token_count > 3
       AND r.is_groundable = true
       AND r.is_gxp_relevant = true
       AND (f.resp_vs_prompt_similarity IS NULL
            OR f.resp_vs_prompt_similarity < 0.80)                       THEN 'high'
      -- Absolute floor alongside the percentile: percent_rank() over a single-row capability is
      -- always 0, so a thin capability would score medium purely by construction.
      -- Observed similarity band is 0.69-0.89; 0.75 is provisional pending calibration.
      WHEN f.similarity_pctile_in_capability < 0.05
       AND f.resp_vs_prompt_similarity < 0.75                            THEN 'medium'
      WHEN f.resp_vs_prompt_similarity IS NULL                           THEN 'unverified'
      ELSE 'low'
    END AS hallucination_risk,
    current_timestamp() AS detected_at
FROM      mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN      mq_gmdf_dev.oil_obs.capability_registry r
       ON r.capability = b.capability AND r.active = true AND r.is_generative = true
       AND r.is_groundable = true
LEFT JOIN mq_gmdf_dev.oil_obs.faithfulness_scores    f ON f.id = b.id
LEFT JOIN mq_gmdf_dev.oil_obs.v_ungrounded_tokens    d ON d.id = b.id
WHERE b.success = true
  AND b.is_blank_output        = false
  AND b.is_credential_fastfail = false
  AND b.called_at >= GREATEST(
      (SELECT last_processed_ts FROM mq_gmdf_dev.oil_obs._obs_watermark
       WHERE detector = 'faithfulness_scores'),
      current_timestamp() - INTERVAL 7 DAYS
  );